# Synthetic vs Real vs Real + Synthetic：2025 真實資料測試分析

## tl;dr

- **加入 synthetic 對 real-only 有明顯的描述性改善。**跨 5 個生成模型、2 種費率與 3 個 seeds，real+synthetic 相對 real-only 的平均測試報酬增加 **2.47%**，Sharpe 增加 **0.105**；30 個 seed-level 配對中，報酬有 **23/30** 次較高，Sharpe 有 **21/30** 次較高。
- **但 combined 並沒有穩定超越 synthetic-only。**相對 synthetic-only，平均報酬只增加 **0.23%**、Sharpe 增加 **0.017**；30 個配對中僅 **13/30** 次報酬較高、**14/30** 次 Sharpe 較高。這比較像 real data 提供正則化與風險控制，而不是全面提升報酬。
- **無交易成本時，最佳 combined 是 Real + ARMD (single path)。**平均報酬 **25.79%**、Sharpe **1.094**、最大回撤 **-22.61%**。它的 Sharpe 高於 Buy & Hold (1.054)，但報酬低於 Buy & Hold (28.30%)。
- **含 0.25% 交易成本時，最佳 combined 是 Real + CN-Diff (50 paths)。**平均報酬 **22.82%**、Sharpe **1.022**、最大回撤 **-22.36%**。Sharpe 接近但略低於 Buy & Hold (1.031)，報酬也較低 (27.49%)。
- **結論屬於描述性而非統計定論。**每個條件只有 3 個 seeds，且所有模型共用單一 2025 真實測試年度；模型選擇若依此測試集進行，會有多重比較與 test-set overfitting 風險。

## Context & Methods

### Key Assumptions

- 使用最新且包含 real+synthetic 的完整結果集 `full/b5a9990cf90b`。
- 訓練期為 2024-01-01 至 2024-10-01，checkpoint 以共同的 real 2024 validation 選擇；最後在未參與訓練與選點的 real 2025 測試。
- real+synthetic 並非把資料列直接合併，而是每個 episode 以 50/50 機率抽 real 或該生成器的 synthetic path。
- 比較以相同 seed、相同 fee 配對；主要指標為累積報酬、Sharpe、年化波動、最大回撤與 turnover。
- 所有結果皆為 PPO checkpoint 的 deterministic rollout。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'finrl').is_dir() and (candidate / 'results').is_dir():
            return candidate
    raise RuntimeError('FinRL project root not found')

PROJECT_ROOT = find_project_root(Path.cwd())
RESULT_DIR = PROJECT_ROOT / 'results/ppo_synthetic_vs_real/full/b5a9990cf90b'
TEST_PATH = RESULT_DIR / 'test_metrics.csv'
PERIOD_PATH = RESULT_DIR / 'ppo_period_metrics.csv'
BENCHMARK_PATH = RESULT_DIR / 'buy_and_hold_metrics.csv'
print('Result set:', RESULT_DIR.relative_to(PROJECT_ROOT))

## Data

### 1. Load and validate the formal result set

In [ ]:
test = pd.read_csv(TEST_PATH)
period = pd.read_csv(PERIOD_PATH)
benchmark = pd.read_csv(BENCHMARK_PATH)

METRICS = ['cumulative_return', 'sharpe', 'annualized_volatility', 'max_drawdown', 'turnover']
assert len(test) == 66, f'Expected 66 test rows, got {len(test)}'
assert test.duplicated(['group', 'commission_name', 'seed']).sum() == 0
assert test[METRICS].notna().all().all()
counts = test.groupby(['commission_name', 'group']).size()
assert counts.eq(3).all(), counts[counts.ne(3)]
assert set(test['seed']) == {0, 1, 2}
quality_summary = pd.DataFrame({
    'check': ['test rows', 'unique fee × group cells', 'seeds per cell', 'duplicate keys', 'missing primary metrics'],
    'result': [len(test), len(counts), f'{counts.min()}–{counts.max()}', int(test.duplicated(['group','commission_name','seed']).sum()), int(test[METRICS].isna().sum().sum())],
})
display(quality_summary)

## Key findings with visual evidence

### 2. Build paired combined-minus-baseline comparisons

A positive delta is favorable for return, Sharpe, and max drawdown (less negative drawdown). A negative delta is favorable for volatility and turnover.

In [ ]:
MODEL_LABELS = {
    'armd__single_path': 'ARMD (single path)',
    'cndiff__50_paths': 'CN-Diff (50 paths)',
    'dva__50_paths': 'DVA (50 paths)',
    'dva__mean_of_50_paths': 'DVA (mean of 50 paths)',
    'nsdiff__50_paths': 'NS-Diff (50 paths)',
}
paired_rows = []
for fee in ['no_fee', 'with_fee']:
    fee_rows = test.loc[test['commission_name'].eq(fee)].set_index(['group', 'seed'])
    real = fee_rows.xs('real_trained', level='group')
    for model_name, model_label in MODEL_LABELS.items():
        combined = fee_rows.xs(f'real_synthetic::{model_name}', level='group')
        synthetic = fee_rows.xs(f'synthetic::{model_name}', level='group')
        for baseline_name, baseline in [('real', real), ('synthetic', synthetic)]:
            deltas = combined[METRICS] - baseline[METRICS]
            for seed, row in deltas.iterrows():
                paired_rows.append({
                    'commission_name': fee, 'model_name': model_name,
                    'model_label': model_label, 'baseline': baseline_name,
                    'seed': int(seed),
                    **{f'delta_{metric}': row[metric] for metric in METRICS},
                })
paired = pd.DataFrame(paired_rows)
paired.head()

In [ ]:
favorable_direction = {
    'cumulative_return': 1, 'sharpe': 1, 'annualized_volatility': -1,
    'max_drawdown': 1, 'turnover': -1,
}
overall_rows = []
for baseline in ['real', 'synthetic']:
    subset = paired.loc[paired['baseline'].eq(baseline)]
    for metric in METRICS:
        values = subset[f'delta_{metric}']
        direction = favorable_direction[metric]
        overall_rows.append({
            'baseline': baseline, 'metric': metric,
            'mean_delta': values.mean(),
            'favorable_seed_pairs': int((direction * values > 0).sum()),
            'total_seed_pairs': len(values),
        })
overall_summary = pd.DataFrame(overall_rows)
display(overall_summary)

**Interpretation.** Combined training consistently improves on real-only across most seed-level comparisons, especially on risk-adjusted performance and drawdown. Against synthetic-only, however, wins are close to a coin flip. The added real episodes therefore look more like a stabilizer than a universal performance boost.

In [ ]:
condition = (
    paired.groupby(['commission_name', 'model_label', 'baseline'])
    .agg(
        sharpe_delta_mean=('delta_sharpe', 'mean'),
        sharpe_delta_std=('delta_sharpe', 'std'),
        return_delta_mean=('delta_cumulative_return', 'mean'),
        volatility_delta_mean=('delta_annualized_volatility', 'mean'),
        drawdown_delta_mean=('delta_max_drawdown', 'mean'),
        turnover_delta_mean=('delta_turnover', 'mean'),
        sharpe_wins=('delta_sharpe', lambda values: int((values > 0).sum())),
    )
    .reset_index()
)
display(condition.sort_values(['commission_name', 'baseline', 'sharpe_delta_mean'], ascending=[True, True, False]))

### 3. Sharpe deltas reveal where combined training helps

Bars show the mean paired Sharpe change across three seeds; error bars are ±1 sample standard deviation, not confidence intervals.

In [ ]:
palette = {'real': '#0057B8', 'synthetic': '#E66100'}
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
model_order = list(MODEL_LABELS.values())
y = np.arange(len(model_order))
offsets = {'real': -0.18, 'synthetic': 0.18}
for axis, fee in zip(axes, ['no_fee', 'with_fee']):
    fee_rows = condition.loc[condition['commission_name'].eq(fee)]
    for baseline in ['real', 'synthetic']:
        rows = fee_rows.loc[fee_rows['baseline'].eq(baseline)].set_index('model_label').reindex(model_order)
        axis.barh(
            y + offsets[baseline], rows['sharpe_delta_mean'], height=0.32,
            xerr=rows['sharpe_delta_std'], color=palette[baseline], alpha=0.85,
            label=f'Combined − {baseline}', capsize=3,
        )
    axis.axvline(0, color='#374151', linewidth=1)
    axis.set_title('No fee' if fee == 'no_fee' else '0.25% transaction fee')
    axis.set_xlabel('Mean paired test Sharpe delta (±1 SD)')
    axis.grid(axis='x', alpha=0.2)
axes[0].set_yticks(y, model_order)
axes[0].invert_yaxis()
axes[0].legend(loc='lower right')
fig.suptitle('Real + synthetic training: paired Sharpe change on real 2025 test')
fig.tight_layout()
plt.show()

**Interpretation.** ARMD is the clearest case where adding real episodes raises Sharpe relative to synthetic-only, while also lowering volatility. For DVA and NS-Diff, combined-vs-synthetic effects are mixed and often reverse by fee setting.

### 4. Compare aggregate strategies with the real-only control and Buy & Hold

In [ ]:
aggregate = (
    test.groupby(['commission_name', 'agent'])[METRICS]
    .agg(['mean', 'std'])
)
benchmark_test = benchmark.loc[benchmark['period'].eq('test')].copy()
comparison_rows = []
for fee in ['no_fee', 'with_fee']:
    fee_agg = aggregate.loc[fee].copy()
    combined = fee_agg.loc[fee_agg.index.str.startswith('Real +')].copy()
    combined['strategy'] = combined.index
    for _, row in combined.iterrows():
        comparison_rows.append({
            'fee': fee, 'strategy': row['strategy'], 'type': 'real + synthetic',
            'return_mean': row[('cumulative_return', 'mean')],
            'return_std': row[('cumulative_return', 'std')],
            'sharpe_mean': row[('sharpe', 'mean')],
            'sharpe_std': row[('sharpe', 'std')],
            'max_drawdown_mean': row[('max_drawdown', 'mean')],
        })
    real = fee_agg.loc['Real data (2024)']
    comparison_rows.append({
        'fee': fee, 'strategy': 'Real data (2024)', 'type': 'real-only',
        'return_mean': real[('cumulative_return','mean')], 'return_std': real[('cumulative_return','std')],
        'sharpe_mean': real[('sharpe','mean')], 'sharpe_std': real[('sharpe','std')],
        'max_drawdown_mean': real[('max_drawdown','mean')],
    })
    bh = benchmark_test.loc[benchmark_test['commission_name'].eq(fee)].iloc[0]
    comparison_rows.append({
        'fee': fee, 'strategy': 'Buy & Hold', 'type': 'benchmark',
        'return_mean': bh['cumulative_return'], 'return_std': np.nan,
        'sharpe_mean': bh['sharpe'], 'sharpe_std': np.nan,
        'max_drawdown_mean': bh['max_drawdown'],
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison.sort_values(['fee', 'sharpe_mean'], ascending=[True, False]))

**Interpretation.** The best combined policies narrow the gap to Buy & Hold by improving drawdown and Sharpe, but they generally give up total return. No-fee Real+ARMD slightly exceeds Buy & Hold on Sharpe; under fees, Real+CN-Diff is close but does not exceed it.

## Robustness, limitations, and validation details

### 5. Fee sensitivity

In [ ]:
fee_pivot = test.pivot(index=['group', 'seed'], columns='commission_name', values=METRICS)
fee_delta = fee_pivot.xs('with_fee', axis=1, level=1) - fee_pivot.xs('no_fee', axis=1, level=1)
fee_delta['strategy_family'] = np.select(
    [fee_delta.index.get_level_values('group').str.startswith('real_synthetic::'),
     fee_delta.index.get_level_values('group').str.startswith('synthetic::')],
    ['real + synthetic', 'synthetic-only'], default='real-only',
)
fee_summary = fee_delta.groupby('strategy_family')[METRICS].mean().reset_index()
display(fee_summary)

Combined policies show a smaller average degradation from no-fee to with-fee runs than the other families. This is encouraging, but it is not a pure transaction-cost subtraction: each fee condition trains a separate policy, so the delta includes policy adaptation and seed variation.

### 6. What the design supports—and what it does not

- **Strengths:** common real validation for checkpoint selection, untouched real 2025 test, same PPO architecture/hyperparameters, paired seeds, deterministic rollout, and both no-fee/with-fee scenarios.
- **Small sample:** only 3 seeds per condition. Standard deviations are descriptive; formal significance tests would have very low power.
- **Single market regime:** all conclusions depend on one 2025 test year and five large-cap US stocks.
- **Multiple comparisons:** choosing a winner among 10 synthetic/combined variants using the same test year risks test-set overfitting.
- **Scaler confound:** combined and synthetic-only use the generator-specific scaler, while real-only uses the real-data scaler. Combined-vs-synthetic isolates added real episodes reasonably well; combined-vs-real also changes the scaler.
- **Post-hoc representative seeds:** median-test-Sharpe representatives are useful plots, not an unbiased selection rule. This notebook bases conclusions on all-seed aggregates and paired deltas.
- **Balanced curriculum:** real+synthetic samples the source 50/50 by episode. This tests a source-balanced curriculum, not arbitrary row-level concatenation ratios.

## Takeaways

1. **Keep real+synthetic as an augmentation strategy, not as a guaranteed replacement for synthetic-only.** It is reliably better than real-only, but not reliably better than the corresponding synthetic-only policy.
2. **Prioritize ARMD for no-fee/risk-adjusted experiments and CN-Diff for fee-aware robustness.** These are the clearest combined candidates in this run.
3. **Do not declare a final winner yet.** Expand to at least 10–20 seeds, multiple rolling test windows, and bootstrap confidence intervals by market period.
4. **Run an ablation on the real episode probability** (for example 0%, 25%, 50%, 75%, 100%) and a separate scaler ablation. This will identify whether gains come from real-data exposure, source balance, or normalization.
5. **Lock a model choice before the next untouched test window.** Use 2024 validation for tuning, select one or two candidates, then evaluate once on a new holdout period to control test-set selection bias.

## Further questions

- Are the combined gains concentrated in particular months or volatility regimes within 2025?
- Does the 50/50 episode ratio remain optimal when synthetic path count changes from 1 to 50?
- Would a scaler fitted jointly on real and synthetic train data improve ARMD without leaking future information?
- Do the same rankings hold for broader universes, different asset classes, and rolling out-of-sample years?